# TP optimisation — 00. Données

Ce notebook **complète** le dossier `data/` seuls les fichiers manquants sont créés :

* Diabetes, Iris, Wine : générés hors ligne avec scikit-learn  ;
* MNIST et Fashion-MNIST : **téléchargés via OpenML (Internet requis, quelques minutes, ~150 Mo chacun)**, puis découpés 50k train / 10k val / 10k test officiel.



In [2]:
import sys
print(sys.executable)
import sklearn, pandas, numpy, matplotlib
print("sklearn", sklearn.__version__)
print("pandas", pandas.__version__)

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\python.exe
sklearn 1.9.1
pandas 3.0.6


In [3]:
import os
print("Dossier de travail (cwd) :", os.getcwd())

Dossier de travail (cwd) : c:\Users\lenovo\Desktop\HADRI


In [ ]:
import os
GENERER = True        
AVEC_IMAGES = True   

In [ ]:
from pathlib import Path
DATA_DIR = next((str(p) for p in [Path("data"), Path("../data")] if p.exists()), "data")
print("Données lues dans :", os.path.abspath(DATA_DIR))
os.makedirs(DATA_DIR, exist_ok=True)

Données lues dans : c:\Users\lenovo\Desktop\HADRI\data


In [ ]:
from types import SimpleNamespace
A = SimpleNamespace(no_images=not AVEC_IMAGES, out=DATA_DIR)
if GENERER:
    try:
        """Complète le dossier data/ : ne génère QUE les fichiers manquants
        (les fichiers fournis par l'enseignant ne sont JAMAIS écrasés).
          - Synthétiques (si absents) : nonlinear, spiral3, blobs3
          - Réels (si absents) : Diabetes, Iris, Wine (scikit-learn, hors ligne),
            MNIST et Fashion-MNIST (téléchargés via OpenML : Internet requis).
        Usage : python prepare_data.py [--no-images] [--out data]
        """
        import os
        import numpy as np, pandas as pd
        from sklearn.datasets import load_diabetes, load_iris, load_wine, make_blobs, fetch_openml
        
        a = A
        os.makedirs(a.out, exist_ok=True)
        
        
        def todo(fn):
            """True si le fichier doit être créé (on n'écrase JAMAIS un fichier existant)."""
            if os.path.exists(os.path.join(a.out, fn)):
                print("déjà présent, conservé :", fn); return False
            print("génération              :", fn); return True
        
        
        def path(fn): return os.path.join(a.out, fn)
        
        
        def add_split(n, n_tr, n_va, seed, stratify=None):
            rng = np.random.default_rng(seed)
            sp = np.array(["test"] * n, dtype=object)
            if stratify is None:
                perm = rng.permutation(n)
                sp[perm[:n_tr]] = "train"; sp[perm[n_tr:n_tr + n_va]] = "val"
            else:
                for c in np.unique(stratify):
                    ids = rng.permutation(np.where(stratify == c)[0])
                    k1, k2 = int(round(len(ids) * n_tr / n)), int(round(len(ids) * n_va / n))
                    sp[ids[:k1]] = "train"; sp[ids[k1:k1 + k2]] = "val"
            return sp
        
        
        if todo("regression_nonlinear.csv"):
            rng = np.random.default_rng(0)
            X = rng.uniform(-3, 3, (360, 2))
            y = np.sin(X[:, 0]) + 0.5 * X[:, 1] ** 2 - 0.3 * X[:, 0] * X[:, 1] + rng.normal(0, 0.2, 360)
            df = pd.DataFrame(X, columns=["x1", "x2"]); df["y"] = y
            df["split"] = add_split(360, 216, 72, 1)
            df.to_csv(path("regression_nonlinear.csv"), index=False)
        
        if todo("regression_diabetes.csv"):                     
            df = load_diabetes(as_frame=True, scaled=False).frame.rename(columns={"target": "y"})
            df["split"] = add_split(442, 265, 88, 2)
            df.to_csv(path("regression_diabetes.csv"), index=False)
        
        if todo("classification_spiral3.csv"):                  
            rng = np.random.default_rng(3)
            pts, lab = [], []
            for c in range(3):
                r = np.linspace(0.2, 1, 150)
                th = c * 2 * np.pi / 3 + 4 * r + rng.normal(0, 0.15, 150)
                pts.append(np.c_[r * np.cos(th), r * np.sin(th)]); lab += [c] * 150
            X = np.vstack(pts); lab = np.array(lab)
            df = pd.DataFrame(X, columns=["x1", "x2"]); df["label"] = lab
            df["split"] = add_split(450, 270, 90, 4, stratify=lab)
            df.to_csv(path("classification_spiral3.csv"), index=False)
        
        if todo("classification_iris.csv"):                     
            df = load_iris(as_frame=True).frame.rename(columns={"target": "label"})
            df["split"] = add_split(150, 90, 30, 5, stratify=df["label"].values)
            df.to_csv(path("classification_iris.csv"), index=False)
        
        if todo("clustering_blobs3.csv"):
            X, lab = make_blobs(360, centers=3, n_features=2, cluster_std=1.0, random_state=6)
            df = pd.DataFrame(X, columns=["x1", "x2"]); df["true_label"] = lab
            df.to_csv(path("clustering_blobs3.csv"), index=False)
        
        if todo("clustering_wine.csv"):                         
            df = load_wine(as_frame=True).frame.rename(columns={"target": "true_label"})
            df.to_csv(path("clustering_wine.csv"), index=False)
        
        if not a.no_images:
            for oid, fn in [("mnist_784", "mnist_50k_10k_10k.npz"), ("Fashion-MNIST", "fashion_mnist_50k_10k_10k.npz")]:
                if not todo(fn): continue
                print("  téléchargement OpenML :", oid, "(quelques minutes)")
                b = fetch_openml(oid, version=1, as_frame=False, parser="auto")
                x, yy = b.data.astype(np.uint8), b.target.astype(np.int64)
                perm = np.random.default_rng(7).permutation(60000)
                tr, va = perm[:50000], perm[50000:]
                np.savez_compressed(path(fn), x_train=x[tr], y_train=yy[tr], x_val=x[va], y_val=yy[va],
                                    x_test=x[60000:], y_test=yy[60000:])
        print("Terminé :", sorted(os.listdir(a.out)))
    except Exception as e:
        print('ERREUR :', repr(e))
        print('Si c\'est le téléchargement OpenML : vérifier la connexion Internet, ou copier les .npz manuellement dans', DATA_DIR)

génération              : regression_nonlinear.csv
génération              : regression_diabetes.csv
génération              : classification_spiral3.csv
génération              : classification_iris.csv
génération              : clustering_blobs3.csv
génération              : clustering_wine.csv
génération              : mnist_50k_10k_10k.npz
  téléchargement OpenML : mnist_784 (quelques minutes)
génération              : fashion_mnist_50k_10k_10k.npz
  téléchargement OpenML : Fashion-MNIST (quelques minutes)
Terminé : ['classification_iris.csv', 'classification_spiral3.csv', 'clustering_blobs3.csv', 'clustering_wine.csv', 'fashion_mnist_50k_10k_10k.npz', 'mnist_50k_10k_10k.npz', 'regression_diabetes.csv', 'regression_nonlinear.csv']


In [7]:
FICHIERS = ["regression_nonlinear.csv", "regression_diabetes.csv", "classification_spiral3.csv", "classification_iris.csv",
            "clustering_blobs3.csv", "clustering_wine.csv", "mnist_50k_10k_10k.npz", "fashion_mnist_50k_10k_10k.npz"]
import os
for f in FICHIERS:
    print(("OK       " if os.path.exists(os.path.join(DATA_DIR, f)) else "MANQUANT ") + f)
import numpy as np, pandas as pd
for f in FICHIERS[:6]:
    p_ = os.path.join(DATA_DIR, f)
    if os.path.exists(p_):
        d = pd.read_csv(p_); print(f, d.shape, d["split"].value_counts().to_dict() if "split" in d else "(pas de colonne split)")
for f in FICHIERS[6:]:
    p_ = os.path.join(DATA_DIR, f)
    if os.path.exists(p_):
        z = np.load(p_); print(f, {k: z[k].shape for k in z.files})

OK       regression_nonlinear.csv
OK       regression_diabetes.csv
OK       classification_spiral3.csv
OK       classification_iris.csv
OK       clustering_blobs3.csv
OK       clustering_wine.csv
OK       mnist_50k_10k_10k.npz
OK       fashion_mnist_50k_10k_10k.npz
regression_nonlinear.csv (360, 4) {'train': 216, 'test': 72, 'val': 72}
regression_diabetes.csv (442, 12) {'train': 265, 'test': 89, 'val': 88}
classification_spiral3.csv (450, 4) {'train': 270, 'val': 90, 'test': 90}
classification_iris.csv (150, 6) {'train': 90, 'test': 30, 'val': 30}
clustering_blobs3.csv (360, 3) (pas de colonne split)
clustering_wine.csv (178, 14) (pas de colonne split)
mnist_50k_10k_10k.npz {'x_train': (50000, 784), 'y_train': (50000,), 'x_val': (10000, 784), 'y_val': (10000,), 'x_test': (10000, 784), 'y_test': (10000,)}
fashion_mnist_50k_10k_10k.npz {'x_train': (50000, 784), 'y_train': (50000,), 'x_val': (10000, 784), 'y_val': (10000,), 'x_test': (10000, 784), 'y_test': (10000,)}
